In [ ]:
# 0. Load datafiles from zip
!unzip engaged.zip -d ./engaged
!unzip disengaged.zip -d ./disengaged

In [ ]:
# 1. Install dependencies
!pip install ultralytics
!pip install "numpy<2.0" scikit-learn==1.7.2

In [ ]:
# 2. Force upgrade both torch and ultralytics to clean up hidden bugs
!pip install --upgrade --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install --upgrade ultralytics

In [ ]:
# 3. Load YOLO pose model
import os
import cv2
import numpy as np
from ultralytics import YOLO

model = YOLO('yolov8n-pose.pt')
# model = YOLO('yolo26n-pose.pt')

In [ ]:
# 4. Data extraction (features)
def calculate_distance(pt1, pt2):
    """Calculates Euclidean distance between two 2D points."""
    return np.linalg.norm(pt1 - pt2)

def calculate_point_to_line_distance(p1, p2, p3):
    """Calculates perpendicular distance of point p3 from line formed by p1 and p2."""
    # Line defined by p1 and p2: ax + by + c = 0
    if np.allclose(p1, p2):
        return calculate_distance(p1, p3)
    return np.abs(np.cross(p2 - p1, p1 - p3)) / np.linalg.norm(p2 - p1)

def extract_facial_features(folder_path, label, conf_thresh=0.7):
    X_data = []
    y_data = []

    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)
        results = model(img_path, verbose=False)

        for result in results:
            # Check if keypoints and confidence scores are present
            if result.keypoints is not None and len(result.keypoints.data) > 0:
                # kp shape: [17, 3] -> containing (x, y, confidence)
                kp = result.keypoints.data[0].cpu().numpy()

                if len(kp) >= 5:
                    nose = kp[0]
                    l_eye = kp[1]
                    r_eye = kp[2]
                    l_ear = kp[3]
                    r_ear = kp[4]

                    # Base rule: Nose must be detected
                    if nose[2] < conf_thresh:
                        continue

                    # Initialize engineered features with default/fallback neutral values
                    nose_eyes_offset = -1.0
                    ear_nose_offset = -1.0

                    # 1. Calculate Nose-to-Eyes Horizontal Offset
                    if l_eye[2] > conf_thresh and r_eye[2] > conf_thresh:
                        eye_center_x = (l_eye[0] + r_eye[0]) / 2.0
                        eye_distance = np.abs(l_eye[0] - r_eye[0])

                        if eye_distance > 0:
                            nose_eyes_offset = np.abs(nose[0] - eye_center_x) / eye_distance

                    # 2. Calculate Ear-to-Nose Perpendicular Offset
                    if l_ear[2] > conf_thresh and r_ear[2] > conf_thresh:
                        ear_dist = calculate_distance(l_ear[:2], r_ear[:2])
                        ear_nose_dist = calculate_point_to_line_distance(l_ear[:2], r_ear[:2], nose[:2])

                        if ear_dist > 0:
                            ear_nose_offset = ear_nose_dist / ear_dist

                    # 3. Calculate Nose-Eyeline Ratio
                    l_eye_nose_dist = (l_eye[0] - nose[0])
                    r_eye_nose_dist = (r_eye[0] - nose[0])
                    # l_eye_nose_dist = calculate_distance(l_eye[:2], nose[:2])
                    # r_eye_nose_dist = calculate_distance(r_eye[:2], nose[:2])
                    nose_eyeline_ratio = abs(l_eye_nose_dist/r_eye_nose_dist)

                    # 4. Number of ears visible
                    num_ears = 0
                    if r_ear[2] > 0.8:
                      num_ears+=1
                    if l_ear[2] > 0.8:
                      num_ears+=1

                    # Append engineered features to the end of the array
                    feature_vector = np.array([nose_eyes_offset, ear_nose_offset, nose_eyeline_ratio])
                    # feature_vector = np.array([nose_eyes_offset, ear_nose_offset])

                    X_data.append(feature_vector)
                    y_data.append(label)
                    break

    return np.array(X_data), np.array(y_data)

# Process folders
X_engaged, y_engaged = extract_facial_features('engaged', 1)
X_disengaged, y_disengaged = extract_facial_features('disengaged', 0)

# Combine datasets
X = np.vstack((X_engaged, X_disengaged))
y = np.concatenate((y_engaged, y_disengaged))

print(f"Data extraction complete. Total samples: {X.shape[0]}")
print(f"Feature vector size per sample: {X.shape[1]} (10 Normalized Coordinates + 2 Engineered Offsets)")

In [ ]:
# 5A. GRADIENT BOOSTING
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score
import pickle

# 1. Split your data (using stratify to handle class balancing)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Scale your 12 features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# 3. Train the Gradient Boosting Model
print("Training HistGradientBoosting Classifier...")
gb_classifier = HistGradientBoostingClassifier(
    max_iter=100,          # Total number of sequential trees (boosting rounds)
    learning_rate=0.05,    # Step size shrinkage to prevent overfitting
    max_depth=5,           # Shallows the trees to keep them focused on general patterns
    random_state=42
)
gb_classifier.fit(X_train_scaled, y_train)

# 4. Check the upgrade
y_pred = gb_classifier.predict(X_val_scaled)
print(f"Gradient Boosting Validation Accuracy: {accuracy_score(y_val, y_pred):.4f}")
print(classification_report(y_val, y_pred))

# 5. Export everything as standard .pkl files for your live script
with open('gaze_classifier.pkl', 'wb') as f:
    pickle.dump(gb_classifier, f)

with open('gaze_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Gradient Boosting artifacts saved successfully!")

In [ ]:
# 5B. REFINED RANDOM FOREST
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import joblib

# Split into 80% training and 20% validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale coordinates so they have a mean of 0 and variance of 1
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

# Initialize with your exact winning sweet spot
final_classifier = RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42
)

# Train the final model
final_classifier.fit(X_train, y_train)

# Double check it still rules
final_score = final_classifier.score(X_val, y_val)
print(f"Final Locked-in Accuracy: {final_score * 100:.2f}%")

# Save your model and scaler so your live script can use them!
joblib.dump(final_classifier, 'gaze_classifier.pkl')
joblib.dump(scaler, 'gaze_scaler.pkl')
print("Model and Scaler exported successfully for deployment!")
y_pred = final_classifier.predict(X_val)
print("Random Forest Validation Accuracy:", accuracy_score(y_val, y_pred))
print("\nClassification Report:\n", classification_report(y_val, y_pred, target_names=['Disengaged', 'Engaged']))

In [ ]:
# 5C. BASIC RANDOM FOREST
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Initialize a Random Forest Classifier
classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
classifier.fit(X_train, y_train)

# Evaluate model performance
y_pred = classifier.predict(X_val)
print("Random Forest Validation Accuracy:", accuracy_score(y_val, y_pred))
print("\nClassification Report:\n", classification_report(y_val, y_pred, target_names=['Disengaged', 'Engaged']))


In [ ]:
# 6. (Optional) Test individual predictions (Change feature_vector accordingly)
def predict_gaze(image_path, conf_thresh=0.7):
    results = model(image_path, verbose=False)
    for result in results:
        # Note: Use result.keypoints.data to get the confidence scores (the 3rd value in the tensor)
        if result.keypoints is not None and len(result.keypoints.data) > 0:
            kp = result.keypoints.data[0].cpu().numpy() # shape: [17, 3]

            if len(kp) >= 5:
                nose = kp[0]
                l_eye = kp[1]
                r_eye = kp[2]
                l_ear = kp[3]
                r_ear = kp[4]

                # Base rule matching training loop: Nose must be visible
                if nose[2] < conf_thresh:
                    continue

                # Initialize engineered features with default/fallback neutral values
                nose_eyes_offset = 0.0
                ear_nose_offset = 0.0

                # 1. Calculate Nose-to-Eyes Horizontal Offset
                if l_eye[2] > conf_thresh and r_eye[2] > conf_thresh:
                    eye_center_x = (l_eye[0] + r_eye[0]) / 2.0
                    eye_distance = np.abs(l_eye[0] - r_eye[0])

                    if eye_distance > 0:
                        nose_eyes_offset = np.abs(nose[0] - eye_center_x) / eye_distance

                # 2. Calculate Ear-to-Nose Perpendicular Offset
                if l_ear[2] > conf_thresh and r_ear[2] > conf_thresh:
                    # Euclidean distance between ears
                    ear_dist = np.linalg.norm(l_ear[:2] - r_ear[:2])
                    # Perpendicular distance from nose to ear-line
                    if not np.allclose(l_ear[:2], r_ear[:2]) and ear_dist > 0:
                        ear_nose_dist = np.abs(np.cross(r_ear[:2] - l_ear[:2], l_ear[:2] - nose[:2])) / ear_dist
                        ear_nose_offset = ear_nose_dist / ear_dist

                # 4. Stitch everything together into the 2-feature array
                feature_vector = np.array([nose_eyes_offset, ear_nose_offset])

                # Reshape to a single row matrix (1, 12) for Scikit-Learn
                feature_vector = feature_vector.reshape(1, -1)

                # 5. Scale and Predict!
                scaled_kp = scaler.transform(feature_vector)

                prediction = gb_classifier.predict(scaled_kp)[0]
                probability = gb_classifier.predict_proba(scaled_kp)[0]

                status = "Engaged (Looking)" if prediction == 1 else "Disengaged"
                confidence = probability[prediction] * 100
                print(f"Result: {status} ({confidence:.2f}% confidence)")
                return

    print("No face detected in frame.")

# Test it!
predict_gaze('engaged/ezgif-frame-150.jpg')


In [ ]:
# 7A. PICKLE MODEL DOWNLOAD (for Google Colab)
from google.colab import files

# This triggers a native browser download for your model and scaler files
print("Downloading gaze_classifier.pkl...")
files.download('gaze_classifier.pkl')

print("Downloading gaze_scaler.pkl...")
files.download('gaze_scaler.pkl')

In [ ]:
# 7B. JOBLIB MODEL DOWNLOAD (for Google Colab)
import joblib
from google.colab import files

# 1. Save the trained classifier and the scaler to Colab's temporary disk
joblib.dump(classifier, 'gaze_classifier.joblib')
joblib.dump(scaler, 'gaze_scaler.joblib')
print("Files successfully saved to Colab disk.")

# 2. Trigger automatic browser downloads for both files
files.download('gaze_classifier.joblib')
files.download('gaze_scaler.joblib')
print("Check your browser's download folder!")
